# 🛡️ CIIBS v3.0 — CargoX-ray Dataset Training
## YOLOv8 Fine-tuning on IS2AI/cargoxray (Trucks & Railcar X-rays)

**Dataset**: [IS2AI/cargoxray](https://github.com/IS2AI/cargoxray) — X-ray images of cargo vehicles (trucks, railcars)

**7 Classes**: Various goods detected in cargo X-ray scans

**Key Advantage**: This dataset has real cargo-scale X-ray images (trucks/railcars), making it ideal for our CIIBS cargo screening system.

---

### ⚙️ Setup
1. Open in **Google Colab** → Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. Download `xray_best.pt` at the end

In [ ]:
# ============================================================
# CELL 1: Setup — Install dependencies & verify GPU
# ============================================================
!pip install -q ultralytics gdown

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → GPU")

## 📥 Step 1: Clone CargoX-ray Repository & Download Data

In [ ]:
# ============================================================
# CELL 2: Clone the CargoX-ray repository
# The data is managed via DVC (Data Version Control)
# We need: annotations.json.gz, images.json.gz, and the images
# ============================================================
import os

%cd /content

# Clone the repo (includes annotation files and data scripts)
if not os.path.exists('/content/cargoxray'):
    !git clone --recursive https://github.com/IS2AI/cargoxray.git
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

# Show structure
!find /content/cargoxray -maxdepth 2 -type f | head -30
print("\nData directory:")
!ls -la /content/cargoxray/data/ 2>/dev/null || echo "No data dir yet"

In [ ]:
# ============================================================
# CELL 3: Install DVC & pull the actual image data
# CargoX-ray uses DVC to store large image files remotely
# ============================================================
!pip install -q dvc dvc-gdrive

%cd /content/cargoxray

# Try to pull data via DVC
print("📥 Attempting to pull data via DVC...")
print("(This downloads the actual X-ray images from remote storage)\n")
!dvc pull 2>&1 || echo "\n⚠️ DVC pull may have failed — will try alternative methods"

In [ ]:
# ============================================================
# CELL 4: Check what data we have and explore annotations
# The annotations are in annotations.json.gz:
#   bbox_id, image_id, x, y, width, height, label
#   (x, y, width, height are ALREADY in YOLO format: 0-1 fractions)
# ============================================================
import gzip
import json
import glob
from collections import Counter

# Find the annotations file
ann_files = glob.glob('/content/cargoxray/**/annotations.json*', recursive=True)
img_meta_files = glob.glob('/content/cargoxray/**/images.json*', recursive=True)

print(f"Annotation files: {ann_files}")
print(f"Image metadata files: {img_meta_files}")

# Load annotations
annotations = []
for af in ann_files:
    if af.endswith('.gz'):
        with gzip.open(af, 'rt') as f:
            annotations = json.load(f)
    else:
        with open(af, 'r') as f:
            annotations = json.load(f)
    print(f"\nLoaded {len(annotations)} annotations from {af}")
    if annotations:
        print(f"Sample annotation: {annotations[0]}")
    break

# Load image metadata
image_meta = []
for imf in img_meta_files:
    if imf.endswith('.gz'):
        with gzip.open(imf, 'rt') as f:
            image_meta = json.load(f)
    else:
        with open(imf, 'r') as f:
            image_meta = json.load(f)
    print(f"\nLoaded {len(image_meta)} image records from {imf}")
    if image_meta:
        print(f"Sample image record: {image_meta[0]}")
    break

# Analyze class distribution
if annotations:
    labels = [a.get('label', a.get('class', 'unknown')) for a in annotations]
    label_counts = Counter(labels)
    print(f"\n📊 Class distribution ({len(label_counts)} classes):")
    for label, count in label_counts.most_common():
        print(f"  {label}: {count}")

# Count unique images
if annotations:
    unique_images = set(a.get('image_id') for a in annotations)
    print(f"\nUnique annotated images: {len(unique_images)}")

# Check for actual image files
print("\n🖼️ Looking for image files...")
img_files = glob.glob('/content/cargoxray/**/*.jpg', recursive=True)
img_files += glob.glob('/content/cargoxray/**/*.png', recursive=True)
img_files += glob.glob('/content/cargoxray/**/*.jpeg', recursive=True)
print(f"Found {len(img_files)} image files")
if img_files:
    print(f"Sample: {img_files[0]}")

## 🔄 Step 2: Convert to YOLO Format & Create Dataset Structure

In [ ]:
# ============================================================
# CELL 5: Convert CargoX-ray to proper YOLO training format
# 
# CargoX-ray annotations.json.gz format:
#   {bbox_id, image_id, x, y, width, height, label}
#   x, y = CENTER coordinates (0-1 fraction) ← already YOLO!
#   width, height = box dimensions (0-1 fraction)
#
# We need to:
#   1. Map labels to class indices
#   2. Create images/ and labels/ directories
#   3. Create train/val splits
#   4. Generate data.yaml
# ============================================================
import os
import shutil
import random
import yaml
from pathlib import Path
from collections import Counter, defaultdict

YOLO_ROOT = '/content/cargoxray_yolo'
TRAIN_SPLIT = 0.85  # 85% train, 15% val

# Build class mapping from annotations
all_labels = sorted(set(a.get('label', a.get('class', 'unknown')) for a in annotations))
label_to_idx = {label: idx for idx, label in enumerate(all_labels)}

print(f"Classes ({len(all_labels)}):")
for label, idx in label_to_idx.items():
    count = sum(1 for a in annotations if a.get('label', a.get('class')) == label)
    print(f"  {idx}: {label} ({count} annotations)")

# Group annotations by image_id
img_annotations = defaultdict(list)
for ann in annotations:
    img_id = ann.get('image_id')
    label = ann.get('label', ann.get('class', 'unknown'))
    cls_idx = label_to_idx[label]
    
    # CargoX-ray format: x, y are CENTER, already normalized 0-1
    x = float(ann.get('x', 0))
    y_val = float(ann.get('y', 0))
    w = float(ann.get('width', 0))
    h = float(ann.get('height', 0))
    
    # Clamp values
    x = max(0, min(1, x))
    y_val = max(0, min(1, y_val))
    w = max(0, min(1, w))
    h = max(0, min(1, h))
    
    if w > 0 and h > 0:
        img_annotations[img_id].append(f"{cls_idx} {x:.6f} {y_val:.6f} {w:.6f} {h:.6f}")

print(f"\nTotal images with annotations: {len(img_annotations)}")

# Build image_id -> filepath mapping
img_id_to_path = {}
if image_meta:
    for im in image_meta:
        img_id = im.get('image_id')
        filepath = im.get('filepath', '')
        # Try to find actual file
        candidates = [
            os.path.join('/content/cargoxray', filepath),
            os.path.join('/content/cargoxray/data/cargoxray/images', os.path.basename(filepath)),
            os.path.join('/content/cargoxray/data/cargoxray', filepath),
        ]
        for c in candidates:
            if os.path.exists(c):
                img_id_to_path[img_id] = c
                break

# If image_meta doesn't give us paths, try matching by ID in image files
if not img_id_to_path and img_files:
    print("\nAttempting to match images by filename...")
    for img_f in img_files:
        basename = os.path.splitext(os.path.basename(img_f))[0]
        # Try matching by various ID patterns
        img_id_to_path[basename] = img_f

print(f"Mapped {len(img_id_to_path)} images to file paths")

# If we still have no image paths, list what we can find
if not img_id_to_path:
    print("\n⚠️ Could not map image IDs to files.")
    print("Available image files:")
    for f in img_files[:10]:
        print(f"  {f}")
    print("\nAnnotation image_ids (sample):")
    sample_ids = list(img_annotations.keys())[:10]
    for sid in sample_ids:
        print(f"  {sid}")

In [ ]:
# ============================================================
# CELL 6: Create YOLO dataset structure with train/val split
# ============================================================

# Create directories
for split in ['train', 'val']:
    os.makedirs(os.path.join(YOLO_ROOT, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_ROOT, 'labels', split), exist_ok=True)

# Get list of image IDs that have both annotations AND image files
valid_ids = []
for img_id in img_annotations:
    if img_id in img_id_to_path:
        valid_ids.append(img_id)

# If img_id_to_path is empty but we have images, try alternative approach
if not valid_ids and img_files:
    print("Using alternative image matching strategy...")
    # Match annotations to image files by trying different ID formats
    file_basenames = {os.path.splitext(os.path.basename(f))[0]: f for f in img_files}
    
    for img_id in img_annotations:
        # Try direct match
        str_id = str(img_id)
        if str_id in file_basenames:
            img_id_to_path[img_id] = file_basenames[str_id]
            valid_ids.append(img_id)
            continue
        # Try padded match
        for basename, path in file_basenames.items():
            if str_id in basename or basename in str_id:
                img_id_to_path[img_id] = path
                valid_ids.append(img_id)
                break

print(f"Valid image+annotation pairs: {len(valid_ids)}")

if not valid_ids:
    print("\n❌ No valid image-annotation pairs found.")
    print("This likely means DVC data hasn't been downloaded.")
    print("Attempting to download images directly...")
    
    # Try DVC pull again with verbose
    %cd /content/cargoxray
    !dvc pull -v 2>&1 | tail -20
    
    # Re-check
    img_files = glob.glob('/content/cargoxray/**/*.jpg', recursive=True)
    img_files += glob.glob('/content/cargoxray/**/*.png', recursive=True)
    print(f"\nAfter DVC pull: {len(img_files)} image files found")
else:
    # Perform train/val split
    random.seed(42)
    random.shuffle(valid_ids)
    split_idx = int(len(valid_ids) * TRAIN_SPLIT)
    train_ids = valid_ids[:split_idx]
    val_ids = valid_ids[split_idx:]
    
    print(f"Train: {len(train_ids)} images")
    print(f"Val:   {len(val_ids)} images")
    
    # Copy/symlink images and write label files
    for split_name, split_ids in [('train', train_ids), ('val', val_ids)]:
        for img_id in split_ids:
            src_img = img_id_to_path[img_id]
            ext = os.path.splitext(src_img)[1]
            dst_img = os.path.join(YOLO_ROOT, 'images', split_name, f"{img_id}{ext}")
            dst_lbl = os.path.join(YOLO_ROOT, 'labels', split_name, f"{img_id}.txt")
            
            # Symlink image (saves disk space)
            if not os.path.exists(dst_img):
                try:
                    os.symlink(os.path.abspath(src_img), dst_img)
                except:
                    shutil.copy2(src_img, dst_img)
            
            # Write label
            with open(dst_lbl, 'w') as f:
                f.write('\n'.join(img_annotations[img_id]))
    
    print("\n✅ Dataset prepared!")
    for split in ['train', 'val']:
        n_imgs = len(os.listdir(os.path.join(YOLO_ROOT, 'images', split)))
        n_lbls = len(os.listdir(os.path.join(YOLO_ROOT, 'labels', split)))
        print(f"  {split}: {n_imgs} images, {n_lbls} labels")

In [ ]:
# ============================================================
# CELL 7: Create data.yaml for YOLOv8
# ============================================================

data_config = {
    'path': YOLO_ROOT,
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(all_labels),
    'names': all_labels
}

data_yaml_path = os.path.join(YOLO_ROOT, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ data.yaml created:")
!cat {data_yaml_path}

In [ ]:
# ============================================================
# CELL 8: Visualize sample training images with boxes
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22',
           '#34495e', '#d35400', '#c0392b']

train_img_dir = os.path.join(YOLO_ROOT, 'images', 'train')
train_lbl_dir = os.path.join(YOLO_ROOT, 'labels', 'train')

if os.path.exists(train_img_dir) and os.listdir(train_img_dir):
    img_files_vis = [f for f in os.listdir(train_img_dir) if f.endswith(('.jpg','.png','.jpeg'))]
    
    # Filter to images with annotations
    annotated = []
    for f in img_files_vis[:100]:
        lbl = os.path.join(train_lbl_dir, os.path.splitext(f)[0] + '.txt')
        if os.path.exists(lbl) and os.path.getsize(lbl) > 0:
            annotated.append(f)
    
    if annotated:
        samples = random.sample(annotated, min(6, len(annotated)))
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        for ax, fname in zip(axes.flat, samples):
            img = Image.open(os.path.join(train_img_dir, fname))
            w, h = img.size
            ax.imshow(img)
            
            lbl_path = os.path.join(train_lbl_dir, os.path.splitext(fname)[0] + '.txt')
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls_id = int(parts[0])
                        cx, cy, bw, bh = [float(x) for x in parts[1:]]
                        x1 = (cx - bw/2) * w
                        y1 = (cy - bh/2) * h
                        rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                            linewidth=2, edgecolor=COLORS[cls_id % len(COLORS)], facecolor='none')
                        ax.add_patch(rect)
                        cls_name = all_labels[cls_id] if cls_id < len(all_labels) else f'cls{cls_id}'
                        ax.text(x1, y1-3, cls_name, color=COLORS[cls_id % len(COLORS)],
                                fontsize=9, fontweight='bold',
                                bbox=dict(facecolor='black', alpha=0.7, pad=1))
            ax.set_title(fname[:30], fontsize=9)
            ax.axis('off')
        
        plt.suptitle('CargoX-ray Training Samples', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('/content/cargoxray_samples.png', dpi=150)
        plt.show()
    else:
        print("No annotated images found for visualization")
else:
    print("Training directory not ready yet")

## 🎯 Step 3: Train YOLOv8 on CargoX-ray

**Strategy**: Mirror their proven hyperparameters (from `train_hyp.yaml`) but upgrade to YOLOv8.

Key adaptations:
- `imgsz=1024` (their original, cargo X-rays are large)
- `mosaic=1.0`, `mixup=0.2` (proven effective for X-ray)
- `degrees=3`, `shear=3` (conservative rotation for cargo)
- `hsv_v=0.25`, `hsv_h/s=0` (X-rays don't have natural color)

In [ ]:
# ============================================================
# CELL 9: Train YOLOv8s on CargoX-ray
# Hyperparameters adapted from IS2AI's train_hyp.yaml
# ============================================================
from ultralytics import YOLO

# Determine batch size based on image size and GPU memory
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_mem >= 15:  # T4, A100
        BATCH_SIZE = 8   # imgsz=1024 is memory hungry
        IMGSZ = 1024
    else:
        BATCH_SIZE = 8
        IMGSZ = 640
else:
    BATCH_SIZE = 4
    IMGSZ = 640

print(f"Training config: batch={BATCH_SIZE}, imgsz={IMGSZ}")

# Load pretrained YOLOv8s
model = YOLO('yolov8s.pt')

# Train with CargoX-ray optimized hyperparameters
results = model.train(
    data=data_yaml_path,
    
    # Core
    epochs=80,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    
    # Optimizer (from their train_hyp.yaml)
    optimizer='AdamW',
    lr0=0.001,            # Slightly lower than their 0.01 (better for transfer)
    lrf=0.01,             # Their lrf=0.2, we go lower for finer convergence
    momentum=0.937,       # Same as theirs
    weight_decay=0.0005,  # Same
    warmup_epochs=3,      # Same
    warmup_momentum=0.8,  # Same
    warmup_bias_lr=0.1,   # Same
    
    # Loss weights (adapted from theirs)
    box=7.5,              # Their box=0.05 (YOLOv5 scale), equiv ~7.5 in v8
    cls=0.5,              # Their cls=0.3, slightly higher for v8
    dfl=1.5,              # Distribution focal loss
    
    # Data augmentation (from their train_hyp.yaml)
    hsv_h=0.0,            # No hue shift (X-rays are grayscale/pseudocolor)
    hsv_s=0.0,            # No saturation shift
    hsv_v=0.25,           # Brightness only (their value)
    degrees=3.0,          # Their rotation (conservative for cargo)
    translate=0.1,        # Same
    scale=0.5,            # Same
    shear=3.0,            # Same
    perspective=0.0,      # Same (no perspective warp)
    flipud=0.0,           # Same (no vertical flip — cargo orientation matters)
    fliplr=0.5,           # Same
    mosaic=1.0,           # Same
    mixup=0.2,            # Same
    copy_paste=0.0,       # Same
    
    # Early stopping
    patience=20,
    
    # Compute
    workers=4,
    device=0,
    amp=True,
    
    # Checkpoints + eval
    save=True,
    save_period=10,
    val=True,
    plots=True,
    
    # Naming
    project='/content/cargoxray_training',
    name='yolov8s_cargoxray_v1',
    exist_ok=True,
    
    # Misc
    verbose=True,
    seed=42,
    deterministic=True,
    close_mosaic=15,  # Disable mosaic last 15 epochs for fine-tuning
)

print("\n\n✅ Training complete!")

## 📊 Step 4: Comprehensive Evaluation

In [ ]:
# ============================================================
# CELL 10: Load best model and evaluate
# ============================================================
import glob

best_path = '/content/cargoxray_training/yolov8s_cargoxray_v1/weights/best.pt'
if not os.path.exists(best_path):
    candidates = glob.glob('/content/cargoxray_training/**/best.pt', recursive=True)
    if candidates:
        best_path = candidates[0]
    else:
        print("❌ No best.pt found")

print(f"Best model: {best_path}")
best_model = YOLO(best_path)

# Full evaluation
print("\n📊 Running evaluation...")
val_results = best_model.val(
    data=data_yaml_path,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    conf=0.25,
    iou=0.6,
    device=0,
    plots=True,
    save_json=True,
    verbose=True,
)

# Detailed metrics
mp = val_results.box.mp
mr = val_results.box.mr
f1 = 2 * mp * mr / (mp + mr + 1e-6)

print(f"\n{'='*60}")
print(f"  CARGOXRAY EVALUATION RESULTS")
print(f"{'='*60}")
print(f"  mAP@0.5:       {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95:  {val_results.box.map:.4f}")
print(f"  Precision:      {mp:.4f}")
print(f"  Recall:         {mr:.4f}")
print(f"  F1 Score:       {f1:.4f}")
print(f"  FNR:            {1-mr:.4f}")
print(f"  FPR:            {1-mp:.4f}")

print(f"\n  Per-Class AP@0.5:")
cnames = val_results.names
for i, ap in enumerate(val_results.box.ap50):
    name = cnames.get(i, f'class_{i}')
    bar = '█' * int(ap * 30) + '░' * (30 - int(ap * 30))
    print(f"    {name:20s}: {ap:.4f}  {bar}")

print(f"\n  Speed:")
print(f"    Inference: {val_results.speed['inference']:.1f}ms")
print(f"    Total: {sum(val_results.speed.values()):.1f}ms")
print(f"    FPS: {1000/max(sum(val_results.speed.values()), 1):.1f}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 11: Display all training plots
# ============================================================
from IPython.display import Image as IPImage, display

train_dir = '/content/cargoxray_training/yolov8s_cargoxray_v1'

for fname, title in [
    ('results.png', '📈 Training Curves'),
    ('confusion_matrix_normalized.png', '📊 Normalized Confusion Matrix'),
    ('F1_curve.png', '📉 F1 vs Confidence'),
    ('PR_curve.png', '📉 Precision-Recall Curve'),
    ('P_curve.png', '📉 Precision vs Confidence'),
    ('R_curve.png', '📉 Recall vs Confidence'),
    ('val_batch0_pred.png', '🖼️ Validation Predictions'),
]:
    path = os.path.join(train_dir, fname)
    if not os.path.exists(path):
        found = glob.glob(f'{train_dir}/**/{fname}', recursive=True)
        path = found[0] if found else None
    if path and os.path.exists(path):
        print(f"\n{title}")
        display(IPImage(filename=path, width=800))

In [ ]:
# ============================================================
# CELL 12: Visual inference on validation images
# ============================================================
import matplotlib.pyplot as plt
import cv2

val_img_dir = os.path.join(YOLO_ROOT, 'images', 'val')
if os.path.exists(val_img_dir) and os.listdir(val_img_dir):
    test_imgs = [os.path.join(val_img_dir, f) for f in os.listdir(val_img_dir)
                 if f.endswith(('.jpg', '.png'))]
    samples = random.sample(test_imgs, min(8, len(test_imgs)))
    
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    for ax, img_path in zip(axes.flat, samples):
        result = best_model(img_path, conf=0.25, imgsz=IMGSZ, verbose=False)[0]
        annotated = result.plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        n = len(result.boxes) if result.boxes is not None else 0
        ax.set_title(f'{os.path.basename(img_path)[:20]} ({n} det)', fontsize=9)
        ax.axis('off')
    
    plt.suptitle('YOLOv8s CargoX-ray Predictions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/cargoxray_predictions.png', dpi=150)
    plt.show()

## 🚀 Step 5: Export & Download Model

In [ ]:
# ============================================================
# CELL 13: Package model for CIIBS integration
# ============================================================
import shutil
import json

export_dir = '/content/ciibs_cargoxray_model'
os.makedirs(export_dir, exist_ok=True)

# Copy best weights
shutil.copy2(best_path, os.path.join(export_dir, 'xray_best.pt'))

# Copy last weights
last_path = best_path.replace('best.pt', 'last.pt')
if os.path.exists(last_path):
    shutil.copy2(last_path, os.path.join(export_dir, 'xray_last.pt'))

# Save metrics
metrics = {
    'model': 'YOLOv8s-CargoXray',
    'dataset': 'IS2AI/cargoxray',
    'classes': list(cnames.values()),
    'num_classes': len(cnames),
    'training_images': len(train_ids) if 'train_ids' in dir() else 0,
    'val_images': len(val_ids) if 'val_ids' in dir() else 0,
    'mAP_50': float(val_results.box.map50),
    'mAP_50_95': float(val_results.box.map),
    'precision': float(mp),
    'recall': float(mr),
    'f1': float(f1),
    'fnr': float(1 - mr),
    'fpr': float(1 - mp),
    'per_class_ap50': {cnames[i]: float(ap) for i, ap in enumerate(val_results.box.ap50)},
    'speed_ms': {
        'preprocess': float(val_results.speed['preprocess']),
        'inference': float(val_results.speed['inference']),
        'postprocess': float(val_results.speed['postprocess'])
    },
    'training_config': {
        'epochs': 80,
        'batch_size': BATCH_SIZE,
        'image_size': IMGSZ,
        'optimizer': 'AdamW',
        'lr': 0.001,
        'source_hyperparams': 'IS2AI/cargoxray/params/train_hyp.yaml'
    }
}

with open(os.path.join(export_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

# Copy training plots
for p in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png',
          'confusion_matrix_normalized.png']:
    src = os.path.join(train_dir, p)
    if os.path.exists(src):
        shutil.copy2(src, export_dir)

# Create zip
shutil.make_archive('/content/ciibs_cargoxray_model', 'zip', export_dir)

print("\n✅ Export complete!")
for f in sorted(os.listdir(export_dir)):
    sz = os.path.getsize(os.path.join(export_dir, f)) / 1e6
    print(f"  {f}: {sz:.1f} MB")

zip_sz = os.path.getsize('/content/ciibs_cargoxray_model.zip') / 1e6
print(f"\n📦 Download: /content/ciibs_cargoxray_model.zip ({zip_sz:.1f} MB)")

In [ ]:
# ============================================================
# CELL 14: Download to your machine
# ============================================================
try:
    from google.colab import files
    print("📥 Starting download...")
    files.download('/content/ciibs_cargoxray_model.zip')
    print("\n✅ Download started!")
    print("After download:")
    print("  1. Extract xray_best.pt from the zip")
    print("  2. Place it in your ciibs/ project folder")
    print("  3. Run: python integrate_model.py xray_best.pt")
except:
    print("Download manually: /content/ciibs_cargoxray_model.zip")

In [ ]:
# ============================================================
# CELL 15: Final summary
# ============================================================
print(f"""
{'='*65}
  🛡️ CIIBS — CargoX-ray Training Report
{'='*65}

  Dataset:     IS2AI/cargoxray
  Classes:     {len(cnames)} ({', '.join(cnames.values())})
  Model:       YOLOv8s
  Image Size:  {IMGSZ}

  📊 Metrics:
     mAP@0.5:       {val_results.box.map50:.4f}
     mAP@0.5:0.95:  {val_results.box.map:.4f}
     Precision:      {mp:.4f}
     Recall:         {mr:.4f}
     F1:             {f1:.4f}

  ⚡ Speed:
     Inference:      {val_results.speed['inference']:.1f}ms
     FPS:            {1000/max(sum(val_results.speed.values()), 1):.1f}

  📦 Model: xray_best.pt ({os.path.getsize(best_path)/1e6:.1f} MB)

{'='*65}
  Integration:
  1. Download ciibs_cargoxray_model.zip
  2. Extract xray_best.pt to ciibs/ folder
  3. Run: python integrate_model.py xray_best.pt
  4. Restart server: PORT=5001 python app.py
{'='*65}
""")